# Branch Smoke Checks: APH/PPH Split

This notebook does quick, rough validation checks for branch changes around the APH/PPH split and hemorrhage logic.

Checks covered:
1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence
4. HemoglobinRiskEffect: low hemoglobin simulants have higher APH/PPH incidence
5. Exploratory anemia burden check stratified by hemorrhage severity

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    SIMULATION_EVENT_NAMES,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 60_000

SIMULATION_STEPS = [
    SIMULATION_EVENT_NAMES.FIRST_TRIMESTER_ANC,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_SCREENING,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_INTERVENTION,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_VISIT_TIMING,
    SIMULATION_EVENT_NAMES.ULTRASOUND,
    SIMULATION_EVENT_NAMES.ANTEPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.DELIVERY_FACILITY,
    SIMULATION_EVENT_NAMES.AZITHROMYCIN_ACCESS,
    SIMULATION_EVENT_NAMES.MISOPROSTOL_ACCESS,
    SIMULATION_EVENT_NAMES.CPAP_ACCESS,
    SIMULATION_EVENT_NAMES.ACS_ACCESS,
    SIMULATION_EVENT_NAMES.ANTIBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.PROBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.OBSTRUCTED_LABOR,
    SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.MATERNAL_SEPSIS,
    SIMULATION_EVENT_NAMES.RESIDUAL_MATERNAL_DISORDERS,
    SIMULATION_EVENT_NAMES.ABORTION_MISCARRIAGE_ECTOPIC_PREGNANCY,
    SIMULATION_EVENT_NAMES.MORTALITY,
    SIMULATION_EVENT_NAMES.EARLY_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.LATE_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.POSTPARTUM_DEPRESSION,
]

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

def run_to_step(sim: InteractiveContext, step_name: str):
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

## 1) APH/PPH incidence and severity vs artifact (rough)

In [ ]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

## 2) Only severe hemorrhage cases die from hemorrhage

In [ ]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[APH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[PPH_SEVERITY_COL] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

## 3) Misoprostol affects postpartum hemorrhage incidence

In [ ]:
def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
    ])

    out = []
    overall = float(pop[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
    out.append({'scenario': scenario, 'group': 'overall', 'pph_incidence': overall, 'n': len(pop)})

    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) == 0:
            continue
        out.append({
            'scenario': scenario,
            'group': f'misoprostol_available={value}',
            'pph_incidence': float(sub[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(sub),
        })

    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    if len(home):
        out.append({
            'scenario': scenario,
            'group': 'home_only',
            'pph_incidence': float(home[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(home),
        })

    return pd.DataFrame(out)

baseline_pph = pph_summary_for_scenario('baseline')
miso_vv_pph = pph_summary_for_scenario('misoprostol_vv')

check4 = pd.concat([baseline_pph, miso_vv_pph], ignore_index=True)
check4

In [ ]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])

## 4) HemoglobinRiskEffect modifies APH/PPH incidence

Higher hemoglobin is protective (RR < 1 at high exposure). Expected direction: low-hemoglobin tertile has higher APH and PPH incidence than the high-hemoglobin tertile.

In [ ]:
# Reuse sim from check 1 (already run to PPH step)
hgb_pop = sim.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

aph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

hgb_pop['hgb_tertile'] = pd.qcut(hgb_pop[COLUMNS.HEMOGLOBIN_EXPOSURE], q=3, labels=['low', 'mid', 'high'])

rows = []
for tertile in ['low', 'mid', 'high']:
    t_mask = hgb_pop['hgb_tertile'] == tertile
    rows.append({
        'hgb_tertile': tertile,
        'aph_incidence': float(hgb_pop.loc[t_mask & aph_elig, COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool).mean()),
        'pph_incidence': float(hgb_pop.loc[t_mask & pph_elig, COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool).mean()),
    })

check4_hgb = pd.DataFrame(rows)

low_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'aph_incidence'].iloc[0]
high_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'aph_incidence'].iloc[0]
low_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'pph_incidence'].iloc[0]
high_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'pph_incidence'].iloc[0]

check4_hgb['directional_pass'] = None
check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'directional_pass'] = (low_aph > high_aph) and (low_pph > high_pph)

check4_hgb

## 5) Exploratory anemia check stratified by hemorrhage severity

This is a provisional check (not a strict test): compare average anemia disability weight by highest hemorrhage severity at the end of the run.
Expected direction: higher burden for severe > moderate > none.

In [ ]:
def anemia_status_from_hgb(hgb: pd.Series) -> pd.Series:
    return (
        pd.cut(
            hgb,
            bins=[-np.inf] + ANEMIA_THRESHOLDS,
            labels=['severe', 'moderate', 'mild'],
            right=False,
        )
        .astype('object')
        .fillna('not_anemic')
    )

anemia_dw_map = {
    'severe': 0.149,
    'moderate': 0.052,
    'mild': 0.004,
    'not_anemic': 0.0,
}

APH_SEVERITY_COL = f"{COLUMNS.ANTEPARTUM_HEMORRHAGE}_severity"
PPH_SEVERITY_COL = f"{COLUMNS.POSTPARTUM_HEMORRHAGE}_severity"

sim_anemia = make_sim('baseline')
run_to_step(sim_anemia, SIMULATION_EVENT_NAMES.POSTPARTUM_DEPRESSION)
pop = sim_anemia.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    APH_SEVERITY_COL,
    PPH_SEVERITY_COL,
])

severity_rank = {HEMORRHAGE_SEVERITY.NONE: 0, HEMORRHAGE_SEVERITY.MODERATE: 1, HEMORRHAGE_SEVERITY.SEVERE: 2}
aph_rank = pop[APH_SEVERITY_COL].map(severity_rank).fillna(0).astype(int)
pph_rank = pop[PPH_SEVERITY_COL].map(severity_rank).fillna(0).astype(int)
max_rank = np.maximum(aph_rank.values, pph_rank.values)
rank_to_label = {0: 'none', 1: 'moderate', 2: 'severe'}
pop['max_hemorrhage_severity'] = pd.Series(max_rank, index=pop.index).map(rank_to_label)

pop['anemia_status'] = anemia_status_from_hgb(pop[COLUMNS.HEMOGLOBIN_EXPOSURE])
pop['anemia_dw'] = pop['anemia_status'].map(anemia_dw_map)

severity_summary = (
    pop.groupby('max_hemorrhage_severity')
    .agg(
        n=('anemia_dw', 'size'),
        mean_anemia_dw=('anemia_dw', 'mean'),
        severe_anemia_share=('anemia_status', lambda x: (x == 'severe').mean()),
        moderate_or_worse_share=('anemia_status', lambda x: x.isin(['severe', 'moderate']).mean()),
    )
    .sort_index()
)

severity_summary